# 07 – EC3D One-Shot Retrieval (NO UNKNOWN)

**Versione paper-aligned**: 11 classi, senza Unknown.

In [1]:
USE_NO_UNKNOWN = True
N_CLASSES = 11
RESULTS_SUBDIR = 'NO_UNKNOWN'

import sys
from pathlib import Path
import pickle
import json
import random
from datetime import datetime

import numpy as np
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score, f1_score

ROOT_DIR = Path('..').resolve()
sys.path.insert(0, str(ROOT_DIR))
sys.path.insert(0, str(ROOT_DIR.parent))

from pose_encoder.twostream_stgcn_plus import TwoStreamSTGCNPlusEncoder
from utils import load_ec3d

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

In [2]:
DATA_DIR = ROOT_DIR.parent / 'data' / 'EC3D'
LOGS_DIR = ROOT_DIR.parent / 'logs'
RESULTS_DIR = ROOT_DIR.parent / 'results' / 'ec3d' / RESULTS_SUBDIR
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

ec3d_data = load_ec3d(DATA_DIR, no_unknown=USE_NO_UNKNOWN)
sequences = ec3d_data['sequences']
labels = np.array(ec3d_data['labels'])
train_indices = ec3d_data['train_indices']
test_indices = ec3d_data['test_indices']
NUM_CLASSES = ec3d_data['num_classes']

train_labels = labels[train_indices]
test_labels = labels[test_indices]
print(f'Train: {len(train_indices)}, Test: {len(test_indices)}, Classes: {NUM_CLASSES}')

Train: 274, Test: 88, Classes: 11


In [3]:
# Load encoder
with open(LOGS_DIR / 'best_epoch.txt') as f:
    BEST_EPOCH = int(f.read().strip())

pose_encoder = TwoStreamSTGCNPlusEncoder(input_dim=3, hidden_channels=[64,128,256,256], output_dim=128, num_nodes=25, dropout=0.1, fusion_dropout=0.3)
pose_encoder.load_state_dict(torch.load(LOGS_DIR / f'pose_encoder_epoch{BEST_EPOCH}.pt', map_location=device))
pose_encoder.to(device).eval()

TwoStreamSTGCNPlusEncoder(
  (pos_blocks): ModuleList(
    (0): STGCNPlusBlock(
      (gcn): GCNLayer(
        (conv): Conv2d(3, 64, kernel_size=(1, 1), stride=(1, 1))
        (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (tcn): TCNLayer(
        (conv): Conv2d(64, 64, kernel_size=(9, 1), stride=(1, 1), padding=(4, 0))
        (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (relu): ReLU(inplace=True)
    )
    (1): STGCNPlusBlock(
      (gcn): GCNLayer(
        (conv): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1))
        (bn): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (tcn): TCNLayer(
        (conv): Conv2d(128, 128, kernel_size=(9, 1), stride=(1, 1), padding=(4, 0))
        (bn): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (dropou

In [4]:
def encode_pose(seq, device, max_len=150):
    seq = np.transpose(seq, (0, 2, 1))
    T = seq.shape[0]
    if T < max_len:
        seq = np.concatenate([seq, np.zeros((max_len - T, 25, 3), dtype=np.float32)], axis=0)
    elif T > max_len:
        seq = seq[np.linspace(0, T-1, max_len).astype(int)]
    seq_t = torch.from_numpy(seq.astype(np.float32)).unsqueeze(0).to(device)
    with torch.no_grad():
        emb = pose_encoder(seq_t)
        emb = F.normalize(emb, p=2, dim=1)
    return emb.squeeze(0).cpu().numpy()

# Precompute embeddings
train_embeddings = np.vstack([encode_pose(sequences[i], device) for i in tqdm(train_indices, desc='Train')])
test_embeddings = np.vstack([encode_pose(sequences[i], device) for i in tqdm(test_indices, desc='Test')])
print(f'Train emb: {train_embeddings.shape}, Test emb: {test_embeddings.shape}')

Train:   0%|          | 0/274 [00:00<?, ?it/s]

Test:   0%|          | 0/88 [00:00<?, ?it/s]

Train emb: (274, 128), Test emb: (88, 128)


In [5]:
# One-shot evaluation over multiple runs
N_RUNS = 10
accuracies = []

for run_seed in range(N_RUNS):
    np.random.seed(run_seed)
    random.seed(run_seed)
    
    # Sample 1 example per class
    support_indices = []
    support_labels = []
    for cid in range(NUM_CLASSES):
        class_mask = train_labels == cid
        class_indices = np.where(class_mask)[0]
        if len(class_indices) > 0:
            selected = np.random.choice(class_indices)
            support_indices.append(selected)
            support_labels.append(cid)
    
    support_labels = np.array(support_labels)
    support_embeddings = train_embeddings[support_indices]
    
    # Classify test set
    sims = cosine_similarity(test_embeddings, support_embeddings)
    preds = support_labels[sims.argmax(axis=1)]
    acc = accuracy_score(test_labels, preds)
    accuracies.append(acc)

accuracies = np.array(accuracies)
print(f'One-Shot Accuracy: {accuracies.mean():.4f} +/- {accuracies.std():.4f}')

One-Shot Accuracy: 0.4295 +/- 0.0717


In [6]:
# Save results
results = {
    'experiment': 'one_shot_retrieval', 'version': 'NO_UNKNOWN', 'num_classes': NUM_CLASSES,
    'timestamp': datetime.now().isoformat(),
    'results': {
        'mean_accuracy': float(accuracies.mean()),
        'std_accuracy': float(accuracies.std()),
        'n_runs': N_RUNS,
        'all_accuracies': accuracies.tolist()
    }
}

with open(RESULTS_DIR / 'one_shot_metrics.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f'Results saved to {RESULTS_DIR}')

Results saved to /home/giov/Scrivania/Tesi/pose-text-feedback-thesis/results/ec3d/NO_UNKNOWN
